# Clase 6 — Teoría: de despliegue a feedback: pon el modelo a trabajar

**Duración estimada: 90 minutos**

## Objetivos de aprendizaje

- Describir qué implica desplegar un modelo y quiénes participan.
- Explicar cómo el feedback de los usuarios reales alimenta la mejora continua del modelo.
- Convertir la predicción de un modelo en un informe legible para alguien que no es data scientist.

Este notebook acompaña al [capítulo 6 del manual](../manual/07_despliegue_feedback.md). Ahí tienes la explicación completa con analogías; aquí verás el contenido resumido en esquema, más una versión funcionando del ejemplo de código.

## 1. Desplegar no es "terminar"

Un modelo que solo vive en un notebook no genera ningún valor real. **Desplegar** significa ponerlo a prueba en el mundo real — a veces primero con un grupo limitado de usuarios o en un entorno de prueba — para construir confianza antes de aplicarlo a gran escala.

### ¿Quién participa en un despliegue?

- **Dueño de la solución** (*solution owner*): decide si el modelo está listo y responde por que resuelva el problema de negocio.
- **Marketing**: comunica qué hace el modelo y por qué importa a quienes tienen que empezar a confiar en él.
- **Desarrolladores de aplicaciones**: construyen la herramienta real (web, app) donde el modelo va a vivir.
- **Administración de TI**: garantiza que todo corra de forma segura, estable, con los datos correctos llegando a tiempo.

Un modelo técnicamente excelente, sin este equipo alrededor, se queda encerrado en un notebook.

## 2. El caso CHF: de árbol de decisión a herramienta clínica

Recuerda el caso central del manual: predecir qué pacientes con **insuficiencia cardíaca congestiva** (CHF) tienen alto riesgo de reingresar dentro de los 30 días tras el alta.

El primer paso del despliegue no fue técnico, fue de **traducción**: el equipo de negocio tradujo los resultados del modelo para que el personal clínico pudiera identificar pacientes de alto riesgo y diseñar acciones de intervención.

La aplicación pedida durante la etapa de requisitos de negocio tenía que:

1. Generar datos del paciente durante toda su estancia hospitalaria.
2. Prepararlos automáticamente en el formato que el modelo necesita.
3. Puntuar a cada paciente cerca del momento del alta, casi en tiempo real.
4. Mostrarse en una app fácil de usar desde el navegador de una tablet.

El despliegue también incluyó **formación al personal clínico** y colaboración con **TI** y **DBA** (administradores de bases de datos) para construir los procesos de seguimiento — el paso que después permitiría medir el feedback.

## 3. Segundo caso real: el mapa de Cognos para diabetes juvenil

Mismo enfoque técnico (árbol de decisión), problema clínico distinto: riesgo de hospitalización en pacientes con **diabetes juvenil**, desplegado como aplicación interactiva con **Cognos** (herramienta de IBM para informes y paneles interactivos).

Tres niveles de vista del mismo modelo:

| Nivel | Qué muestra | Para quién |
|---|---|---|
| Mapa nacional | Overview de riesgo de hospitalización en todo el país, análisis interactivo por condiciones del paciente | Gestión de salud pública |
| Resumen por subgrupo | Riesgo dentro de un nodo concreto del árbol, combinación de condiciones del subgrupo | Personal clínico que quiere entender un patrón |
| Resumen individual | Riesgo predicho + historial clínico de un paciente concreto | El médico que atiende a ese paciente |

Es el mismo modelo, pero la forma de presentarlo cambia según quién lo mira y para qué lo necesita.

## 4. Feedback: por qué la metodología es un ciclo

Una vez desplegado, el modelo entra en la etapa de **Feedback**: los usuarios reales devuelven información sobre cómo funciona en la práctica, y eso permite refinarlo y evaluar su impacto real.

> "Cuanto más sabes, más quieres saber." — John Rollins

*(Analogía propia, no de IBM: el feedback es como un termostato — no ajusta una vez y se olvida, mide, compara y corrige todo el tiempo.)*

### El ciclo de feedback del caso CHF

1. **Definir el proceso de revisión** — responsabilidad de ejecutivos de gestión clínica.
2. **Seguimiento de pacientes con intervención** — se registran sus resultados de reingreso.
3. **Medir la efectividad** — *sin* grupo de control vs. tratamiento, por razones éticas. Comparación de tasas de reingreso **antes vs. después**.
4. **Revisión al cabo de un año** de implementación.
5. **Refinar el modelo** — incluida la posibilidad de incorporar por fin los datos farmacéuticos que se habían aplazado en la etapa de recolección (clase 3).
6. **Revisar y refinar las acciones de intervención** en sí mismas.
7. **Redesplegar** — el ciclo de feedback continúa mientras el programa exista.

**Idea clave:** el feedback puede llevarte de vuelta a *cualquier* etapa anterior de la metodología (no solo a repetir el modelado) — en el caso CHF, empujó a reconsiderar una decisión tomada en la etapa de recolección de datos.

## 5. De probabilidades a informe legible

No existe código oficial de IBM para estas dos etapas — los datos hospitalarios del caso CHF no están disponibles públicamente. Lo que sigue es un **ejemplo ligero e ilustrativo, de elaboración propia**, que retoma el árbol de decisión entrenado en la clase anterior sobre el dataset de recetas de cocina, y simula el mismo tipo de "traducción" que hizo el equipo de negocio del caso CHF: convertir la salida numérica de un modelo en un informe que cualquier persona pueda leer.

In [ ]:
# Simulamos el tipo de salida que da predict_proba() de un
# DecisionTreeClassifier de scikit-learn ya entrenado (como el que
# construiste en la clase anterior sobre el dataset de recetas):
# una lista de tuplas (nombre, cocina predicha, confianza de la predicción).
predicciones = [
    ("receta_1", "thai", 0.82),
    ("receta_2", "indian", 0.65),
    ("receta_3", "mexican", 0.94),
    ("receta_4", "french", 0.51),
]

def generar_informe(predicciones):
    """Convierte predicciones crudas del modelo en frases legibles
    para alguien sin conocimientos tecnicos, igual que el equipo de
    negocio del caso CHF tradujo probabilidades de reingreso en un
    informe de riesgo legible para un clinico."""
    lineas = []
    for nombre, cocina, confianza in predicciones:
        porcentaje = round(confianza * 100)
        lineas.append(
            f"{nombre.replace('_', ' ').capitalize()}: se predice cocina "
            f"{cocina.upper()} con un {porcentaje}% de confianza."
        )
    return "\n".join(lineas)

print(generar_informe(predicciones))

### La misma idea, con pandas

Como ya usaste `pandas` en clases anteriores, aquí tienes la misma traducción pero organizada en un DataFrame — el tipo de tabla resumen que un dueño de solución o un gestor clínico podría revisar de un vistazo.

In [ ]:
import pandas as pd

df_predicciones = pd.DataFrame(
    predicciones, columns=["item", "cocina_predicha", "confianza"]
)

# Nueva columna: el mismo informe legible, fila por fila.
df_predicciones["informe"] = df_predicciones.apply(
    lambda fila: f"Se predice {fila['cocina_predicha'].upper()} "
                 f"con un {round(fila['confianza'] * 100)}% de confianza.",
    axis=1,
)

df_predicciones

## Puntos clave de esta clase

- Desplegar no es terminar el proyecto: es ponerlo a prueba en el mundo real, primero quizá con un grupo limitado.
- Un despliegue necesita al dueño de la solución, marketing, desarrolladores de aplicaciones y TI — cada uno cubre algo que el data scientist no puede cubrir solo.
- El caso CHF y el caso Cognos de diabetes juvenil muestran el mismo tipo de modelo (árbol de decisión) desplegado de formas distintas según la audiencia.
- El feedback cierra el ciclo de la metodología: el valor del modelo depende de incorporarlo bien, y puede obligarte a volver a cualquier etapa anterior.
- En el caso CHF, por razones éticas, la efectividad se midió comparando tasas de reingreso antes/después, sin grupo de control.
- Convertir una predicción numérica en un informe legible es, literalmente, la tarea que hizo posible que el caso CHF funcionara fuera del notebook.

Sigue con el [notebook de práctica de esta clase](../notebooks_practica/clase06_practica_despliegue_feedback.ipynb).